# Z Image Turbo Bulk Studio for Kaggle

A low-VRAM, sequential bulk image generator built around **`Tongyi-MAI/Z-Image-Turbo`**.

**Features**

- Single prompt or one-prompt-per-line `.txt` upload.
- Social-media aspect-ratio presets.
- Sequential generation to avoid VRAM spikes.
- Model CPU offload with sequential-offload fallback.
- Responsive Stop button checked before and after every image.
- ZIP export with a CSV manifest.
- Privacy-focused image metadata cleanup: GPS, prompts, local paths, and generation metadata are not written into exported images.
- Adult-content safeguards: no minors, age-ambiguous subjects, non-consensual intimate imagery, or real-person sexual impersonation.

> The cleanup stage is for privacy and interoperability. It does **not** fabricate camera metadata, disguise AI origin, or bypass platform provenance, moderation, or AI-detection systems. Follow the rules of the platform where you publish.

> Kaggle GPU memory varies. The default **Memory Saver** profile generates at 768px on the long side and can upscale the clean output for export. Native 1024px generation is available but may require a P100/A100 or more aggressive offloading.

In [ ]:
# Install current Diffusers support for Z Image Turbo.
# Kaggle: enable Internet in Notebook options, then select a GPU accelerator.
!pip install -q -U git+https://github.com/huggingface/diffusers.git transformers accelerate safetensors sentencepiece ipywidgets pillow pandas tqdm

In [ ]:
import os, re, io, csv, time, zipfile, threading, shutil, gc
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone

import torch
import pandas as pd
from PIL import Image
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

WORKDIR = Path('/kaggle/working/z_image_turbo_bulk')
OUTDIR = WORKDIR / 'images'
ZIP_PATH = WORKDIR / 'z_image_turbo_export.zip'
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Tongyi-MAI/Z-Image-Turbo'

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('WARNING: enable a Kaggle GPU before loading the model.')

In [ ]:
# UI theme and safe generation controls
STYLE = """
<style>
.zstudio {font-family: Inter,system-ui,sans-serif; background:linear-gradient(135deg,#0d1021,#17142d 55%,#24163a); color:#f4f1ff; padding:24px; border-radius:22px; margin:8px 0 16px; box-shadow:0 12px 40px #08091388;}
.zstudio h1 {font-size:30px; margin:0 0 5px; letter-spacing:-.03em;}
.zstudio p {color:#c9c2e8; margin:4px 0;}
.zbadge {display:inline-block; border:1px solid #806cff77; color:#cfc7ff; border-radius:999px; padding:5px 10px; margin:12px 5px 0 0; font-size:12px;}
.notice {background:#261d39; border-left:4px solid #a78bfa; padding:10px 12px; border-radius:8px; color:#ddd6fe; margin-top:14px;}
</style>
<div class='zstudio'>
<h1>Z Image Turbo · Bulk Studio</h1>
<p>Fast, sequential, low-VRAM generation for Kaggle.</p>
<span class='zbadge'>6B model</span><span class='zbadge'>8-step Turbo</span><span class='zbadge'>ZIP export</span><span class='zbadge'>Privacy cleanup</span>
<div class='notice'>Use only lawful, consensual adult content. Do not generate minors, age-ambiguous subjects, non-consensual intimate imagery, or sexualized real-person impersonations.</div>
</div>
"""
display(HTML(STYLE))

PROMPT_SAFETY_PATTERNS = [
    r'\b(child|kid|minor|underage|preteen|toddler|baby|infant)\b',
    r'\b(young[- ]looking|schoolgirl|schoolboy|teen\s*(girl|boy)?|barely legal)\b',
    r'\b(non[- ]consensual|without consent|revenge porn|upskirt|hidden camera)\b',
    r'\b(real person|celebrity|public figure)\b.*\b(nude|nsfw|sex|explicit)\b',
]

def safety_reason(prompt: str):
    p = prompt.lower()
    for pat in PROMPT_SAFETY_PATTERNS:
        if re.search(pat, p):
            return 'Blocked by the notebook safety guard: minor/age ambiguity, non-consensual content, or sexualized real-person impersonation.'
    return None

ASPECTS = {
    'Instagram square · 1:1': (768, 768, 1080, 1080),
    'Instagram portrait · 4:5': (768, 960, 1080, 1350),
    'Instagram story/reel · 9:16': (768, 1360, 1080, 1920),
    'Facebook landscape · 1.91:1': (960, 512, 1200, 630),
    'Facebook portrait · 4:5': (768, 960, 1080, 1350),
    'YouTube thumbnail · 16:9': (960, 544, 1280, 720),
    'X/Twitter landscape · 16:9': (960, 544, 1600, 900),
    'Pinterest portrait · 2:3': (768, 1152, 1000, 1500),
    'Kaggle memory saver · 1:1': (640, 640, 1024, 1024),
}

prompt_box = widgets.Textarea(value='A cinematic editorial portrait in soft morning light, textured fabric, natural skin, 35mm composition', description='Prompt:', layout=widgets.Layout(width='100%', height='92px'))
file_upload = widgets.FileUpload(accept='.txt', multiple=False, description='Upload .txt')
use_bulk = widgets.Checkbox(value=False, description='Use uploaded prompts instead of the text box')
aspect = widgets.Dropdown(options=list(ASPECTS), value='Kaggle memory saver · 1:1', description='Aspect:')
num_images = widgets.IntSlider(value=1, min=1, max=100, step=1, description='Per prompt:', continuous_update=False)
seed_box = widgets.IntText(value=12345, description='Base seed:')
format_box = widgets.Dropdown(options=['PNG (lossless)', 'JPEG (smaller ZIP)'], value='PNG (lossless)', description='Format:')
export_native = widgets.Checkbox(value=False, description='Export at social preset size (Lanczos upscale)')
model_offload = widgets.Dropdown(options=['model_cpu_offload (faster)', 'sequential_cpu_offload (lowest VRAM)'], value='model_cpu_offload (faster)', description='Offload:')
steps_box = widgets.IntSlider(value=8, min=8, max=8, description='Steps:', disabled=True)
run_btn = widgets.Button(description='Generate', button_style='success', icon='play')
stop_btn = widgets.Button(description='Stop after current image', button_style='warning', icon='stop')
zip_btn = widgets.Button(description='Build ZIP', button_style='info', icon='archive')
status = widgets.Output(layout={'border':'1px solid #38304f','padding':'10px'})

controls = widgets.VBox([
    prompt_box, widgets.HBox([file_upload, use_bulk]),
    widgets.HBox([aspect, format_box]),
    widgets.HBox([num_images, seed_box]),
    widgets.HBox([export_native, model_offload]),
    widgets.HBox([run_btn, stop_btn, zip_btn]), status
])
display(controls)

In [ ]:
# Load Z Image Turbo lazily. This cell does not download weights until you click Generate.
pipe = None
PIPE_LOCK = threading.Lock()
stop_event = threading.Event()
worker_thread = None


def choose_dtype():
    # Ampere+ can use BF16. T4/P100 profiles use FP16 to avoid unsupported BF16 kernels.
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability()
    return torch.bfloat16 if major >= 8 else torch.float16


def load_pipeline(offload_label=None):
    global pipe
    if pipe is not None:
        return pipe
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU not detected. Enable a Kaggle GPU accelerator.')
    from diffusers import ZImagePipeline
    dtype = choose_dtype()
    with PIPE_LOCK:
        if pipe is None:
            print(f'Loading {MODEL_ID} with {dtype} ...')
            pipe = ZImagePipeline.from_pretrained(
                MODEL_ID,
                torch_dtype=dtype,
                low_cpu_mem_usage=False,
                cache_dir='/kaggle/working/hf_cache'
            )
            offload_label = offload_label or model_offload.value
            if offload_label.startswith('sequential'):
                pipe.enable_sequential_cpu_offload()
            else:
                pipe.enable_model_cpu_offload()
            if hasattr(pipe, 'vae'):
                if hasattr(pipe.vae, 'enable_slicing'): pipe.vae.enable_slicing()
                if hasattr(pipe.vae, 'enable_tiling'): pipe.vae.enable_tiling()
            print('Pipeline ready.')
    return pipe


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def uploaded_text():
    if not file_upload.value:
        return ''
    item = next(iter(file_upload.value.values())) if isinstance(file_upload.value, dict) else file_upload.value[0]
    data = item['content'] if isinstance(item, dict) else item.content
    return data.decode('utf-8', errors='replace')


def get_prompts():
    if use_bulk.value:
        raw = uploaded_text()
        if not raw.strip():
            raise ValueError('Bulk mode is enabled, but no .txt file was uploaded.')
        prompts = [line.strip() for line in raw.splitlines() if line.strip()]
    else:
        prompts = [prompt_box.value.strip()] if prompt_box.value.strip() else []
    if not prompts:
        raise ValueError('Enter a prompt or upload a text file.')
    return prompts


def safe_filename(text, index):
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', text).strip('_')[:55] or 'prompt'
    return f'{index:04d}_{slug}'


def save_clean_image(image, path, fmt):
    # Re-encode from pixels so EXIF, PNG text chunks, prompts, GPS, and local paths are not carried forward.
    image = image.convert('RGB')
    if fmt.startswith('JPEG'):
        image.save(path, format='JPEG', quality=95, subsampling=0, optimize=True, exif=b'')
    else:
        image.save(path, format='PNG', optimize=True)


def make_zip(manifest_rows):
    if ZIP_PATH.exists(): ZIP_PATH.unlink()
    manifest = pd.DataFrame(manifest_rows)
    manifest_path = WORKDIR / 'manifest.csv'
    manifest.to_csv(manifest_path, index=False)
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTDIR.iterdir()):
            if p.is_file(): z.write(p, arcname=f'images/{p.name}')
        z.write(manifest_path, arcname='manifest.csv')
    return ZIP_PATH


def worker():
    global worker_thread
    try:
        prompts = get_prompts()
        gen_w, gen_h, out_w, out_h = ASPECTS[aspect.value]
        ext = 'jpg' if format_box.value.startswith('JPEG') else 'png'
        rows, counter = [], 0
        with status:
            clear_output(wait=True)
            print(f'Queued {len(prompts)} prompt(s) × {num_images.value} image(s).')
            print(f'Generation: {gen_w}×{gen_h}; export: {out_w}×{out_h if export_native.value else gen_h}px')
        local_pipe = load_pipeline(model_offload.value)
        for p_idx, prompt in enumerate(prompts, start=1):
            reason = safety_reason(prompt)
            if reason:
                rows.append({'index': counter + 1, 'status': 'blocked', 'prompt': prompt, 'reason': reason})
                with status: print(f'Blocked prompt {p_idx}: {reason}')
                continue
            for j in range(num_images.value):
                if stop_event.is_set():
                    with status: print('Stop requested. No new image will start.')
                    raise StopIteration
                counter += 1
                seed = int(seed_box.value) + counter - 1
                generator = torch.Generator(device='cuda').manual_seed(seed)
                with status: print(f'Generating {counter} · prompt {p_idx}/{len(prompts)} · seed {seed}')
                with torch.inference_mode():
                    result = local_pipe(
                        prompt=prompt,
                        width=gen_w,
                        height=gen_h,
                        num_inference_steps=8,
                        guidance_scale=0.0,
                        generator=generator,
                    )
                image = result.images[0]
                if export_native.value and (image.width, image.height) != (out_w, out_h):
                    image = image.resize((out_w, out_h), Image.Resampling.LANCZOS)
                path = OUTDIR / f'{safe_filename(prompt, counter)}.{ext}'
                save_clean_image(image, path, format_box.value)
                rows.append({'index': counter, 'status': 'ok', 'file': path.name, 'prompt': prompt, 'seed': seed, 'generation_width': gen_w, 'generation_height': gen_h, 'export_width': image.width, 'export_height': image.height, 'created_utc': datetime.now(timezone.utc).isoformat()})
                with status:
                    display(image)
                clear_memory()
        if rows:
            make_zip(rows)
            with status: print(f'Finished. ZIP ready: {ZIP_PATH}')
    except StopIteration:
        if rows:
            make_zip(rows)
        with status: print('Generation stopped safely after the current image.')
    except Exception as exc:
        with status: print('ERROR:', repr(exc))
        raise
    finally:
        stop_event.clear()
        worker_thread = None


def on_generate(_):
    global worker_thread
    if worker_thread and worker_thread.is_alive():
        with status: print('A generation run is already active.')
        return
    stop_event.clear()
    worker_thread = threading.Thread(target=worker, daemon=True)
    worker_thread.start()


def on_stop(_):
    stop_event.set()
    with status: print('Stop requested. The current diffusion pass will finish, then generation will stop.')


def on_zip(_):
    with status:
        if ZIP_PATH.exists():
            print(f'ZIP: {ZIP_PATH}')
            display(HTML(f'<a href="/files/{ZIP_PATH}" download>Download ZIP</a>'))
        else:
            print('No ZIP exists yet. Generate at least one image first.')

run_btn.on_click(on_generate)
stop_btn.on_click(on_stop)
zip_btn.on_click(on_zip)

## Practical Kaggle memory notes

- **T4/P100:** begin with `Kaggle memory saver · 1:1`, `model_cpu_offload`, and one image per prompt. Use `sequential_cpu_offload` if loading fails.
- **P100:** the notebook automatically selects FP16 because BF16 support is not reliable on older GPUs.
- **A100:** BF16 is selected automatically; 1024px generation is more practical, but bulk generation remains sequential.
- Do not increase batch size. The notebook deliberately generates one image at a time.
- If a run stops, already completed images remain in `/kaggle/working/z_image_turbo_bulk/images/` and are included in the ZIP created before stopping.
- The Stop button cannot safely interrupt a CUDA kernel halfway through denoising; it stops immediately after the current image and prevents the next image from starting.